# 决策树

- 决策树是一种用"一连串是非问题"来做预测的监督学习模型。
- 从根节点开始,每个内部节点问一个关于特征的问题(比如"年龄 > 30?"),根据答案走向不同分支,最后到达叶节点
- 叶节点给出预测结果。
- 它既能做分类(预测类别),也能做回归(预测数值)

- 根节点——树的起点,包含全部训练样本
- 内部节点——存放一个分裂条件(特征 j,阈值 s),把样本分到子节点
- 叶节点——不再分裂的终点,存放最终预测
- 深度——根到最远叶子的边数,代表一条预测路径最多问几个问题
- 叶子数——等于划分出的区域数,是模型复杂度最直接的度量

## 损失函数
- 训练决策树，就是找出一组问题，让树在训练集上的损失尽量小
- 损失——信息混乱程度（信息熵）的变化量
> - **_基尼不纯度 Gini impurity_**
> > 1. $G = 1 - \sum{p_i^2}$——有放回取出两次的类别不同的概率——基尼不纯度越低说明数据越整齐，信息熵越低
> > 2. **_加权不纯度_**——$G_{split} = \frac{n_l}{n}G_{left} + \frac{n_r}{n}G_{right}$
> > 3. 选择一个决策分界点后，数据被分成两份，现在的总基尼不纯度就是加权不纯度——目的就是尽可能降低基尼不纯度

- 一个例子
- 输入：[[1,5],[2,3],[3,4],[4,1]]
- 输出：[0,0,1,1]

In [18]:
import torch
x = torch.tensor([[1,5],[2,3],[3,4],[4,1]])
y = torch.tensor([0,0,1,1])
num_classes = y.max(dim = -1).values.item() + 1

In [19]:


def best_split(x: torch.Tensor, y: torch.Tensor, num_classes: int) -> dict:
    """
    x 的形状：n*k——n个样本，k种特征
    y——1 * n——n个样本
    num_classes——分类数
    """
    n = len(y)
    counts = torch.nn.functional.one_hot(y, num_classes).to(torch.float32)
    best=None

    for feature in range(x.shape[1]):
        # 当前特征的取值
        values = torch.unique(x[:, feature])

        if len(values) < 2:
            continue

        thresholds = (values[:-1] + values[1:])/2

        # a * b，a种阈值，b个样本，值为是否取到左边
        go_left=(x[:, feature] <= thresholds[:, None]).to(torch.float32)

        # counts 形状 b * num_classes，
        # 乘出来的结果就是 a * num_classes，不同阈值下，各种类别的取值个数
        left = go_left @ counts
        # counts.sum(dim=0)是每种类别的数量，形状为1 * b
        right = counts.sum(dim=0)-left

        n_left = left.sum(dim=1)
        n_right = right.sum(dim=1)

        p_left = left / n_left[:, None]
        p_right = right / n_right[:, None]
        gini_left = 1 - (p_left ** 2).sum(dim=1)
        gini_right = 1 - (p_right ** 2).sum(dim=1)

        impurity = (n_left * gini_left + n_right * gini_right) / n

        # argmin 在并列时返回第一个，也就是较小的阈值；严格小于才替换，保证并列时保留较小的特征下标
        i = torch.argmin(impurity)
        if best is None or impurity[i] < best["impurity"]:
            best = {"feature": feature, "threshold": thresholds[i].item(), "impurity": impurity[i].item()}
    return best



In [22]:
# 观察数据可知，特征0的取值分别是1,2,3,4，设置2.5这个阈值，刚好把前两个归类为0，后两个归类为1
res = best_split(x, y, num_classes)
print(f"The best decision choice is depending on feature:{res['feature']} with threshold:{res['threshold']}")

The best decision choice is depending on feature:0 with threshold:2.5
